In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import gc
import math
import json
import joblib
import numpy as np
import pandas as pd
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, rdMolDescriptors, Descriptors, Crippen, Lipinski
from rdkit.Chem import rdFingerprintGenerator

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.decomposition import PCA

TRAIN_CSV = "ML_standardized_smiles.csv"      
LIB_CSV   = "smiles_database.csv"             
SMILES_COL = "smiles"                        # smiles 列名
Y_COL = "liver_avg"                          # 肝标签列名（训练集）
OUTDIR = "feat_out"
os.makedirs(OUTDIR, exist_ok=True)

FP_BITS = 1024
FP_RADIUS = 2

# 库特征生成模式：
LIB_MODE = "FAST_2D"          # "FAST_2D"（默认） or "FULL_3D"（全库3D，极慢）
REFINE_TOPK_3D = 0            # >0 则对库的前 topK 行补 3D（你也可以传入 topK 的 index 列表）
CHUNK_SIZE = 5000             # 库分块大小（避免内存炸）

# 物理特征选择 & PCA
TOPK_PHYS = 43                # MI 选物理特征数量
CORR_PRUNE_THR = 0.95         # 相关性去冗余阈值
FP_PCA_VAR = 0.98             # PCA保留方差
FP_PCA_MAXDIM = 40            # PCA最大维度
RANDOM_STATE = 42

print("OUTDIR:", OUTDIR)


In [ ]:
class LNPFeatureExtractor:
    """
    统一入口：
      compute_features(smiles_list, use_3d=True/False) -> (df_phys, X_fp_uint8)
    2D物理特征：27
    3D物理特征：15（可选）
    另加 has_3d 列（use_3d=True 时）
    """
    def __init__(self, fp_bits=512, fp_radius=2, random_state=42):
        self.fp_bits = int(fp_bits)
        self.fp_radius = int(fp_radius)
        self.random_state = int(random_state)

        # 指纹生成器
        self.fp_gen = rdFingerprintGenerator.GetMorganGenerator(
            radius=self.fp_radius, fpSize=self.fp_bits
        )

        # 特征名
        self.names_2d = self._get_2d_names()
        self.names_3d = self._get_3d_names()
        self.names_full = self.names_2d + self.names_3d

    def __getstate__(self):
        """让对象可被 joblib/pickle：移除不可序列化的 RDKit fp_gen"""
        state = self.__dict__.copy()
        state["fp_gen"] = None
        return state
    
    def __setstate__(self, state):
        """反序列化后重建 fp_gen"""
        self.__dict__.update(state)
        self.fp_gen = rdFingerprintGenerator.GetMorganGenerator(
            radius=int(self.fp_radius),
            fpSize=int(self.fp_bits)
        )

    def _get_2d_names(self):
        return [
            "MolWt","LogP","TPSA","HBD","HBA","RotB","Ring","AromaticRing",
            "HeavyAtomCount","NHOHCount","NOCount","NumAliphaticRings",
            "NumAromaticRings","NumSaturatedRings","NumHeteroatoms",
            "FractionCSP3","MolMR",
            "NumAmideBonds","NumAromaticHeterocycles","NumAromaticCarbocycles",
            "NumAliphaticHeterocycles","NumAliphaticCarbocycles",
            "Chi0v","Chi1v","Chi2v",
            "MaxPartialCharge","MinPartialCharge"
        ]  # 27

    def _get_3d_names(self):
        return [
            "RadiusGyration","Asphericity","Eccentricity","InertialShapeFactor",
            "PMI1","PMI2","PMI3",
            "NPR1","NPR2",
            "LabuteASA","SpherocityIndex",
            "PBF","TPSA_3Dproxy","MolVol","ConfEnergy"
        ]  # 15（部分是proxy/可计算量）

    def _safe(self, fn, default=np.nan):
        try:
            return fn()
        except Exception:
            return default

    def _mol_from_smiles(self, smi):
        try:
            return Chem.MolFromSmiles(str(smi))
        except Exception:
            return None

    def _extract_2d(self, mol):
        # 部分电荷：Gasteiger
        def charges():
            m = Chem.AddHs(mol)
            AllChem.ComputeGasteigerCharges(m)
            qs = []
            for a in m.GetAtoms():
                v = a.GetProp("_GasteigerCharge")
                try:
                    qs.append(float(v))
                except Exception:
                    pass
            if len(qs) == 0:
                return (np.nan, np.nan)
            return (float(np.max(qs)), float(np.min(qs)))

        max_q, min_q = self._safe(charges, (np.nan, np.nan))

        # Chi indices
        chi0v = self._safe(lambda: rdMolDescriptors.CalcChi0v(mol), np.nan)
        chi1v = self._safe(lambda: rdMolDescriptors.CalcChi1v(mol), np.nan)
        chi2v = self._safe(lambda: rdMolDescriptors.CalcChi2v(mol), np.nan)

        ring = Lipinski.RingCount(mol)
        arom_ring = Lipinski.NumAromaticRings(mol)

        out = [
            Descriptors.MolWt(mol),
            Crippen.MolLogP(mol),
            rdMolDescriptors.CalcTPSA(mol),
            Lipinski.NumHDonors(mol),
            Lipinski.NumHAcceptors(mol),
            Lipinski.NumRotatableBonds(mol),
            ring,
            arom_ring,
            mol.GetNumHeavyAtoms(),
            Lipinski.NHOHCount(mol),
            Lipinski.NOCount(mol),
            Lipinski.NumAliphaticRings(mol),
            Lipinski.NumAromaticRings(mol),
            Lipinski.NumSaturatedRings(mol),
            Lipinski.NumHeteroatoms(mol),
            rdMolDescriptors.CalcFractionCSP3(mol),
            Crippen.MolMR(mol),
            rdMolDescriptors.CalcNumAmideBonds(mol),
            rdMolDescriptors.CalcNumAromaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAromaticCarbocycles(mol),
            rdMolDescriptors.CalcNumAliphaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAliphaticCarbocycles(mol),
            chi0v, chi1v, chi2v,
            max_q, min_q
        ]
        return out

    def _embed_3d(self, mol):
        """生成3D构象并UFF优化，失败返回 None"""
        try:
            m = Chem.AddHs(mol)
            params = AllChem.ETKDGv3()
            params.randomSeed = self.random_state
            cid = AllChem.EmbedMolecule(m, params)
            if cid < 0:
                return None
            # UFF优化
            AllChem.UFFOptimizeMolecule(m, maxIters=200)
            return m
        except Exception:
            return None

    def _extract_3d(self, mol3d):
        # 构象能量（UFF）
        conf_energy = self._safe(lambda: AllChem.UFFGetMoleculeForceField(mol3d).CalcEnergy(), np.nan)

        pmi1 = self._safe(lambda: rdMolDescriptors.CalcPMI1(mol3d), np.nan)
        pmi2 = self._safe(lambda: rdMolDescriptors.CalcPMI2(mol3d), np.nan)
        pmi3 = self._safe(lambda: rdMolDescriptors.CalcPMI3(mol3d), np.nan)

        npr1 = self._safe(lambda: rdMolDescriptors.CalcNPR1(mol3d), np.nan)
        npr2 = self._safe(lambda: rdMolDescriptors.CalcNPR2(mol3d), np.nan)

        rog = self._safe(lambda: rdMolDescriptors.CalcRadiusOfGyration(mol3d), np.nan)
        asp = self._safe(lambda: rdMolDescriptors.CalcAsphericity(mol3d), np.nan)
        ecc = self._safe(lambda: rdMolDescriptors.CalcEccentricity(mol3d), np.nan)
        isf = self._safe(lambda: rdMolDescriptors.CalcInertialShapeFactor(mol3d), np.nan)

        asa = self._safe(lambda: rdMolDescriptors.CalcLabuteASA(mol3d), np.nan)
        sph = self._safe(lambda: rdMolDescriptors.CalcSpherocityIndex(mol3d), np.nan)
        pbf = self._safe(lambda: rdMolDescriptors.CalcPBF(mol3d), np.nan)

        # 3D proxy：TPSA仍是2D，这里仅保留作为一致列（也可去掉）
        tpsa_proxy = self._safe(lambda: rdMolDescriptors.CalcTPSA(mol3d), np.nan)

        # 体积（MolVol）
        vol = self._safe(lambda: AllChem.ComputeMolVolume(mol3d), np.nan)

        # 一个简单“3D polar proxy”：PBF/ASA组合（可删，这里保留）
        tpsa_3dproxy = tpsa_proxy

        out = [
            rog, asp, ecc, isf,
            pmi1, pmi2, pmi3,
            npr1, npr2,
            asa, sph,
            pbf, tpsa_3dproxy, vol, conf_energy
        ]
        return out

    def _fp_to_array(self, mol):
        fp = self.fp_gen.GetFingerprint(mol)
        arr = np.zeros((self.fp_bits,), dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    def compute_features(self, smiles_list, use_3d=True):
        feats = []
        fps = []
        has3d = []

        for smi in tqdm(smiles_list, desc=("feat_3d" if use_3d else "feat_2d")):
            mol = self._mol_from_smiles(smi)
            if mol is None:
                feats.append([np.nan] * (len(self.names_full) if use_3d else len(self.names_2d)))
                fps.append(np.zeros((self.fp_bits,), dtype=np.uint8))
                has3d.append(0)
                continue

            f2d = self._extract_2d(mol)

            if use_3d:
                mol3d = self._embed_3d(mol)
                if mol3d is None:
                    f3d = [np.nan] * len(self.names_3d)
                    has3d.append(0)
                else:
                    f3d = self._extract_3d(mol3d)
                    # 如果3D输出含nan也认为失败（可放宽）
                    ok = 0 if any(pd.isna(x) for x in f3d) else 1
                    has3d.append(ok)
                feats.append(f2d + f3d)
            else:
                feats.append(f2d)

            fps.append(self._fp_to_array(mol))

        cols = self.names_full if use_3d else self.names_2d
        df = pd.DataFrame(feats, columns=cols)

        # 填补：2D/3D NaN -> median
        for c in self.names_2d:
            med = df[c].median()
            if pd.isna(med): med = 0.0
            df[c] = df[c].fillna(med)

        if use_3d:
            for c in self.names_3d:
                med = df[c].median()
                if pd.isna(med): med = 0.0
                df[c] = df[c].fillna(med)
            df["has_3d"] = has3d

        df = df.fillna(0.0)
        X_fp = np.array(fps, dtype=np.uint8)
        return df, X_fp


In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
assert SMILES_COL in df_train.columns, f"训练集缺少 {SMILES_COL}"
assert Y_COL in df_train.columns, f"训练集缺少 {Y_COL}"

extractor = LNPFeatureExtractor(fp_bits=FP_BITS, fp_radius=FP_RADIUS, random_state=RANDOM_STATE)

# 训练集：建议做 3D（样本小）
df_phys_train_3d, X_fp_train = extractor.compute_features(df_train[SMILES_COL].tolist(), use_3d=True)

# 2D 子集直接切
df_phys_train_2d = df_phys_train_3d[extractor.names_2d].copy()

# 保存中间结果（强烈建议）
df_phys_train_3d.to_csv(os.path.join(OUTDIR, "train_phys_3d.csv"), index=False)
df_phys_train_2d.to_csv(os.path.join(OUTDIR, "train_phys_2d.csv"), index=False)
np.save(os.path.join(OUTDIR, "train_fp.npy"), X_fp_train)
joblib.dump(extractor, os.path.join(OUTDIR, "feature_extractor.pkl"))

print("Saved train features to", OUTDIR)
print("train_phys_3d:", df_phys_train_3d.shape, "train_fp:", X_fp_train.shape)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.decomposition import PCA

def corr_prune(features, df_scaled, scores_dict, thr=0.95):
    """在候选特征中剔除高度相关（|r|>=thr）的冗余特征，保留得分更高者"""
    keep = []
    for f in features:
        ok = True
        for g in list(keep):
            r = np.corrcoef(df_scaled[f].values, df_scaled[g].values)[0, 1]
            if abs(r) >= thr:
                if scores_dict.get(f, 0) <= scores_dict.get(g, 0):
                    ok = False
                else:
                    keep.remove(g)
                break
        if ok:
            keep.append(f)
    return keep

def fit_preprocessing(df_phys_train, X_fp_train, y_train, topk_phys=30,
                      corr_thr=0.95, fp_var=0.98, fp_maxdim=40, random_state=42):
    """
    在训练集上 fit：
      - 物理特征 StandardScaler
      - MI 筛选 TopK 物理特征 + 相关性去冗余
      - 指纹中心化 + PCA
    返回 pre 字典（供训练/库 transform）
    """
    # y: log1p 与后续一致
    y = np.asarray(y_train).astype(np.float32)
    y_log = np.log1p(np.maximum(y, 0.0))

    # 0) 明确列顺序，避免 transform 时列顺序漂移导致错位
    phys_cols_all = list(df_phys_train.columns)
    df_phys_train = df_phys_train[phys_cols_all].copy()

    print(f"✅ 原始物理特征维度: 样本数={df_phys_train.shape[0]}, 特征数={df_phys_train.shape[1]}")

    # 1) scale phys
    scaler = StandardScaler()
    phys_scaled = pd.DataFrame(
        scaler.fit_transform(df_phys_train.values.astype(np.float32)),
        columns=phys_cols_all
    )

    # 2) MI 选 TopK phys（加 random_state 让结果可复现）
    k = min(int(topk_phys), phys_scaled.shape[1])
    selector = SelectKBest(lambda X, y: mutual_info_regression(X, y, random_state=random_state), k=k)
    selector.fit(phys_scaled.values, y_log)

    imp = pd.DataFrame({
        "Feature": phys_scaled.columns,
        "Score": selector.scores_
    }).sort_values("Score", ascending=False)

    top_features = imp["Feature"].head(k).tolist()
    scores_dict = dict(zip(imp["Feature"], imp["Score"]))

    print(f"✅ MI 筛选 Top{len(top_features)} 物理特征")

    # 3) 相关性去冗余
    top_pruned = corr_prune(top_features, phys_scaled, scores_dict, thr=corr_thr)

    print(f"✅ 相关性去冗余后物理特征数: {len(top_pruned)}")
    print(f"\n✅ 原始指纹特征维度: 样本数={X_fp_train.shape[0]}, 特征数={X_fp_train.shape[1]}")

    # 4) 指纹 PCA（中心化 + PCA）
    X_fp = X_fp_train.astype(np.float32)
    fp_mean = X_fp.mean(axis=0, keepdims=True)  # shape (1, fp_bits)
    X_fp_center = X_fp - fp_mean

    pca = PCA(n_components=fp_var, svd_solver="full", random_state=random_state)
    X_fp_red = pca.fit_transform(X_fp_center)

    # 限制维度上限（若触发则重新 fit，避免后面 transform 维度不一致）
    if X_fp_red.shape[1] > int(fp_maxdim):
        pca = PCA(n_components=int(fp_maxdim), svd_solver="auto", random_state=random_state)
        X_fp_red = pca.fit_transform(X_fp_center)

    print(f"✅ PCA 降维后指纹特征数: {X_fp_red.shape[1]}")
    print(f"\n✅ 最终总输入特征维度 = 物理({len(top_pruned)}) + 指纹PCA({X_fp_red.shape[1]}) = {len(top_pruned) + X_fp_red.shape[1]}")

    pre = {
        "scaler_phys": scaler,
        "selected_phys_features": top_pruned,
        "phys_columns_all": phys_cols_all,
        "fp_mean": fp_mean.squeeze().astype(np.float32),
        "pca_fp": pca,
        "all_feature_cols": top_pruned + [f"FP_PC{i+1}" for i in range(X_fp_red.shape[1])],
        "target_transform": "log1p(liver_avg)",
    }
    return pre

def transform_with_preprocessing(df_phys, X_fp, pre):
    """对任意数据（训练/库）做同样 transform：scale phys -> select -> fp center -> pca -> concat"""
    scaler = pre["scaler_phys"]
    phys_cols_all = pre["phys_columns_all"]

    # ✅ 强制按 fit 时列顺序对齐（避免列顺序变化造成错位）
    df_phys = df_phys[phys_cols_all].copy()

    phys_scaled = pd.DataFrame(
        scaler.transform(df_phys.values.astype(np.float32)),
        columns=phys_cols_all
    )
    phys_sel = phys_scaled[pre["selected_phys_features"]].values.astype(np.float32)

    fp_mean = pre["fp_mean"].reshape(1, -1).astype(np.float32)
    X_fp_center = X_fp.astype(np.float32) - fp_mean
    X_fp_red = pre["pca_fp"].transform(X_fp_center).astype(np.float32)

    return np.hstack([phys_sel, X_fp_red]).astype(np.float32)

print("✅ preprocessing functions ready: fit_preprocessing / transform_with_preprocessing")


In [ ]:
import os, joblib, numpy as np, pandas as pd

# 训练集y
y_train = df_train[Y_COL].values

# 1) fit 2D preprocessing（只用 2D 物理特征）
pre_2d = fit_preprocessing(df_phys_train_2d, X_fp_train, y_train, topk_phys=TOPK_PHYS)
X_final_train_2d = transform_with_preprocessing(df_phys_train_2d, X_fp_train, pre_2d)
print("--------------------------------fit 2D preprocessing---------------------------------")

# 2) fit 3D preprocessing（用 3D 物理特征）
pre_3d = fit_preprocessing(df_phys_train_3d, X_fp_train, y_train, topk_phys=TOPK_PHYS)
X_final_train_3d = transform_with_preprocessing(df_phys_train_3d, X_fp_train, pre_3d)
print("--------------------------------fit 3D preprocessing---------------------------------")

# 3) 保存工件
joblib.dump(pre_2d, os.path.join(OUTDIR, "preprocessing_2d.pkl"))
joblib.dump(pre_3d, os.path.join(OUTDIR, "preprocessing_3d.pkl"))
np.save(os.path.join(OUTDIR, "X_final_train_2d.npy"), X_final_train_2d)
np.save(os.path.join(OUTDIR, "X_final_train_3d.npy"), X_final_train_3d)

# 4) 输出训练表（建议：train_with_Xfinal.csv 绑定为 3D 版本，供 ML_NEW 直接用）
df_train_out_3d = pd.concat(
    [df_train[[c for c in df_train.columns if c != SMILES_COL]].reset_index(drop=True),
     df_train[[SMILES_COL]].reset_index(drop=True),
     pd.DataFrame(X_final_train_3d, columns=pre_3d["all_feature_cols"])],
    axis=1
)
df_train_out_3d.to_csv(os.path.join(OUTDIR, "train_with_Xfinal.csv"), index=False)

# 可选：也保存 2D 版本（方便你训练一个 2D 模型）
df_train_out_2d = pd.concat(
    [df_train[[c for c in df_train.columns if c != SMILES_COL]].reset_index(drop=True),
     df_train[[SMILES_COL]].reset_index(drop=True),
     pd.DataFrame(X_final_train_2d, columns=pre_2d["all_feature_cols"])],
    axis=1
)
df_train_out_2d.to_csv(os.path.join(OUTDIR, "train_with_Xfinal_2d.csv"), index=False)

print("✅ Saved preprocessing_2d/3d + X_final_train_2d/3d + train_with_Xfinal(.csv)")
print("X_final_train_2d:", X_final_train_2d.shape, "X_final_train_3d:", X_final_train_3d.shape)


In [ ]:
import os, gc, numpy as np, pandas as pd, joblib
from numpy.lib.format import open_memmap

LIB_CSV = LIB_CSV if 'LIB_CSV' in globals() else "smiles_database.csv"
SMILES_COL = SMILES_COL if 'SMILES_COL' in globals() else "smiles"
OUTDIR = OUTDIR if 'OUTDIR' in globals() else "feat_out"
CHUNK_SIZE = CHUNK_SIZE if 'CHUNK_SIZE' in globals() else 5000

extractor = joblib.load(os.path.join(OUTDIR, "feature_extractor.pkl"))
pre2 = joblib.load(os.path.join(OUTDIR, "preprocessing_2d.pkl"))

# 只读 smiles 列，避免把整库其它列读进内存
df_lib = pd.read_csv(LIB_CSV, usecols=[SMILES_COL])
assert SMILES_COL in df_lib.columns, f"库缺少 {SMILES_COL}"
n_lib = len(df_lib)
print("Library size:", n_lib)

phys2_csv_path = os.path.join(OUTDIR, "lib_phys_2d.csv")
fp_npy_path    = os.path.join(OUTDIR, "lib_fp.npy")
x2_npy_path    = os.path.join(OUTDIR, "X_final_lib_2d.npy")

# 如果已存在缓存，默认复用（需要重算就手动删文件）
if os.path.exists(fp_npy_path) and os.path.exists(x2_npy_path) and os.path.exists(phys2_csv_path):
    print("✅ Found cached lib features, skip recompute:")
    print(" -", phys2_csv_path)
    print(" -", fp_npy_path)
    print(" -", x2_npy_path)
else:
    # 预先确定最终维度
    x2_dim = len(pre2["selected_phys_features"]) + int(getattr(pre2["pca_fp"], "n_components_", pre2["pca_fp"].components_.shape[0]))
    print("X_final_lib_2d dim =", x2_dim)

    # 流式写入 .npy（不会占用巨量内存）
    fp_mm = open_memmap(fp_npy_path, mode="w+", dtype=np.uint8, shape=(n_lib, int(extractor.fp_bits)))
    x2_mm = open_memmap(x2_npy_path, mode="w+", dtype=np.float32, shape=(n_lib, x2_dim))

    # 物理特征 CSV：分块 append
    if os.path.exists(phys2_csv_path):
        os.remove(phys2_csv_path)
    header_written = False

    for start in range(0, n_lib, CHUNK_SIZE):
        end = min(n_lib, start + CHUNK_SIZE)
        smiles_chunk = df_lib[SMILES_COL].iloc[start:end].tolist()

        df_phys2_chunk, X_fp_chunk = extractor.compute_features(smiles_chunk, use_3d=False)

        # 写入 fp
        fp_mm[start:end, :] = X_fp_chunk.astype(np.uint8)

        # 写入 2D 最终特征
        X2_chunk = transform_with_preprocessing(df_phys2_chunk, X_fp_chunk, pre2)
        x2_mm[start:end, :] = X2_chunk.astype(np.float32)

        # append 物理特征
        df_phys2_chunk.to_csv(
            phys2_csv_path,
            mode="a",
            header=(not header_written),
            index=False
        )
        header_written = True

        if (start // CHUNK_SIZE) % 5 == 0:
            gc.collect()
        print(f"processed {end}/{n_lib}")

    fp_mm.flush()
    x2_mm.flush()

    print("✅ Saved:")
    print(" -", phys2_csv_path)
    print(" -", fp_npy_path)
    print(" -", x2_npy_path)

# quick sanity
X_final_lib_2d = np.load(x2_npy_path, mmap_mode="r")
print("X_final_lib_2d:", X_final_lib_2d.shape, "dtype:", X_final_lib_2d.dtype)
